# XED emotion classification - encoder + LoRA baseline (Group 6)

Started from the BERT notebook of Öhman et al. (2020), rewritten for our setup:
multi-label instead of single label, our own fixed split, XLM-R so romanian works,
LoRA instead of full fine-tuning. Run with `TRAIN_ON` = `en`, `ro`, `both`.
Test set is only used in the last cell.

In [1]:
SEED = 42

MODEL_NAME = "xlm-roberta-base"
TRAIN_ON = "en"          # en / ro / both
EPOCHS = 5
MAX_LEN = 48
BATCH_SIZE = 32
LEARN_RATE = 3e-4        # lora likes a higher lr than full finetuning
WARMUP = 0.06
THRESHOLD = 0.5

USE_LORA = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

RUN_NAME = f"xlmr_{'lora' if USE_LORA else 'full'}_{TRAIN_ON}_s{SEED}"

In [2]:
import json, time, datetime, random, os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_score, recall_score, jaccard_score, hamming_loss, accuracy_score, classification_report

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

# works both from the notebook dir and from repo root
ROOT = Path.cwd()
while not (ROOT / "data" / "splits").exists():
    ROOT = ROOT.parent
SPLITS = ROOT / "data" / "splits"
OUT = ROOT / "runs" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)

cuda NVIDIA GeForce RTX 3050 4GB Laptop GPU


In [3]:
LABELS = ["anger", "anticipation", "disgust", "fear", "joy", "sadness", "surprise", "trust"]

def load(name):
    return [json.loads(l) for l in open(SPLITS / f"{name}.jsonl", encoding="utf-8")]

train_rows, dev_rows, test_rows = load("train"), load("dev"), load("test")

def texts_labels(rows, lang):
    if lang == "both":
        rows = rows + rows
        langs = ["en"] * (len(rows) // 2) + ["ro"] * (len(rows) // 2)
    else:
        langs = [lang] * len(rows)
    X = [r[l] for r, l in zip(rows, langs)]
    y = np.array([r["labels"] for r in rows], dtype=np.float32)
    return X, y

X_train, y_train = texts_labels(train_rows, TRAIN_ON)
X_dev, y_dev = texts_labels(dev_rows, TRAIN_ON)
print("train", len(X_train), "dev", len(X_dev), "test", len(test_rows))
print("label counts train:", dict(zip(LABELS, y_train.sum(0).astype(int).tolist())))

train 3499 dev 1019 test 485
label counts train: {'anger': 940, 'anticipation': 803, 'disgust': 444, 'fear': 526, 'joy': 583, 'sadness': 541, 'surprise': 558, 'trust': 594}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prepare_data(sentences, labels, shuffle=False):
    enc = tokenizer(list(sentences), max_length=MAX_LEN, padding="max_length", truncation=True, return_tensors="pt")
    data = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels))
    sampler = RandomSampler(data) if shuffle else SequentialSampler(data)
    return DataLoader(data, sampler=sampler, batch_size=BATCH_SIZE)

lengths = [len(tokenizer.encode(s)) for s in X_train]
print(f"{np.mean(np.array(lengths) > MAX_LEN):.1%} of train sentences longer than MAX_LEN={MAX_LEN}, max is {max(lengths)}")

train_dataloader = prepare_data(X_train, y_train, shuffle=True)
dev_dataloader = prepare_data(X_dev, y_dev)

0.0% of train sentences longer than MAX_LEN=48, max is 42


In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), problem_type="multi_label_classification"
)

if USE_LORA:
    from peft import LoraConfig, get_peft_model
    lora_cfg = LoraConfig(task_type="SEQ_CLS", r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                          target_modules=["query", "value"])
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

model.to(device);

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,186,568 || all params: 279,236,368 || trainable%: 0.4249


In [6]:
def predict(model, dataloader):
    model.eval()
    probs, gold = [], []
    for input_ids, mask, labels in dataloader:
        with torch.no_grad():
            logits = model(input_ids=input_ids.to(device), attention_mask=mask.to(device)).logits
        probs.append(torch.sigmoid(logits).cpu().numpy())
        gold.append(labels.numpy())
    return np.concatenate(probs), np.concatenate(gold)


def evaluate(probs, gold, verbose=False):
    pred = (probs >= THRESHOLD).astype(int)
    gold = gold.astype(int)
    m = {
        "micro_f1": f1_score(gold, pred, average="micro", zero_division=0),
        "samples_jaccard": jaccard_score(gold, pred, average="samples", zero_division=0),
        "macro_f1": f1_score(gold, pred, average="macro", zero_division=0),
        "micro_precision": precision_score(gold, pred, average="micro", zero_division=0),
        "micro_recall": recall_score(gold, pred, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(gold, pred),
        "exact_match": accuracy_score(gold, pred),
        "empty_pred_rate": float((pred.sum(1) == 0).mean()),
    }
    if verbose:
        for k, v in m.items():
            print(f"{k:16s} {v:.4f}")
        print(classification_report(gold, pred, target_names=LABELS, zero_division=0, digits=3))
    return m

In [ ]:
def train(model, train_dataloader, dev_dataloader, epochs):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARN_RATE)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * total_steps), total_steps)

    stats = []
    best_f1, best_state = -1, None
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for input_ids, mask, labels in train_dataloader:
            model.zero_grad()
            out = model(input_ids=input_ids.to(device), attention_mask=mask.to(device), labels=labels.to(device))
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += out.loss.item()

        probs, gold = predict(model, dev_dataloader)
        dev = evaluate(probs, gold)
        dev_loss = torch.nn.functional.binary_cross_entropy(torch.tensor(probs), torch.tensor(gold)).item()
        stats.append({"epoch": epoch + 1, "train_loss": total_loss / len(train_dataloader), "dev_loss": dev_loss, **dev})
        print(f"epoch {epoch+1}  train loss {stats[-1]['train_loss']:.4f}  dev loss {dev_loss:.4f}  "
              f"dev micro-F1 {dev['micro_f1']:.4f}  jaccard {dev['samples_jaccard']:.4f}  "
              f"({str(datetime.timedelta(seconds=int(time.time()-t0)))})")

        # checkpoint selection on dev micro-F1
        if dev["micro_f1"] > best_f1:
            best_f1 = dev["micro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"best dev micro-F1 {best_f1:.4f} at epoch {max(stats, key=lambda s: s['micro_f1'])['epoch']}")
    return stats

stats = train(model, train_dataloader, dev_dataloader, EPOCHS)

epoch 1  train loss 0.4985  dev loss 0.4639  dev micro-F1 0.0000  jaccard 0.0000  (0:01:16)
epoch 2  train loss 0.4464  dev loss 0.4046  dev micro-F1 0.2200  jaccard 0.1403  (0:01:54)
epoch 3  train loss 0.4202  dev loss 0.3962  dev micro-F1 0.2829  jaccard 0.1893  (0:02:28)


In [ ]:
plt.plot([s["train_loss"] for s in stats], "b-o", label="train")
plt.plot([s["dev_loss"] for s in stats], "g-o", label="dev")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()

plt.plot([s["micro_f1"] for s in stats], "r-o", label="dev micro-F1")
plt.plot([s["samples_jaccard"] for s in stats], "m-o", label="dev jaccard")
plt.xlabel("epoch"); plt.legend(); plt.show()

## Test - run once, at the end

In [ ]:
results = {}
for lang in ["en", "ro"]:
    X_test, y_test = texts_labels(test_rows, lang)
    probs, gold = predict(model, prepare_data(X_test, y_test))
    print(f"===== trained on {TRAIN_ON}, test {lang} =====")
    results[lang] = evaluate(probs, gold, verbose=True)

    with open(OUT / f"preds_test_{lang}.jsonl", "w", encoding="utf-8") as f:
        for r, p in zip(test_rows, probs):
            f.write(json.dumps({"id": r["id"], "probs": [round(float(x), 4) for x in p]}) + "\n")

config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, list))}
config["hardware"] = torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu"
config["checkpoint_rule"] = "best dev micro-F1 over epochs"
json.dump({"config": config, "dev_stats": stats, "test": results}, open(OUT / "results.json", "w"), indent=2)
print("saved to", OUT)